In [1]:
# make autoreload cell
%load_ext autoreload
%autoreload 2



In [2]:
import pandas as pd
from pathlib import Path


In [3]:
resolutie_file = Path("/Users/rikhoekstra/Downloads/resolutions-version_2025-02-20.jsonl.gz")
resolutie_df = pd.read_json(resolutie_file, lines=True, compression="gzip")


In [4]:
len(resolutie_df)
resolutie_df.head()

,id,type,metadata,paragraphs,entities
0,session-3788-num-1-resolution-1,resolution,"{'session_weekday': 'vrijdag', 'session_month'...","[{'id': 'session-3788-num-1-para-2', 'type': '...","[{'entity': {'id': 'P0003911', 'name': 'Spina'..."
1,session-3788-num-1-resolution-10,resolution,"{'session_weekday': 'vrijdag', 'session_month'...","[{'id': 'session-3788-num-1-para-11', 'type': ...","[{'reference': {'layer': 'RES', 'inv': 3788, '..."
2,session-3788-num-1-resolution-11,resolution,"{'session_weekday': 'vrijdag', 'session_month'...","[{'id': 'session-3788-num-1-para-12', 'type': ...","[{'entity': {'id': 'P0005097', 'name': 'Ooster..."
3,session-3788-num-1-resolution-12,resolution,"{'session_weekday': 'vrijdag', 'session_month'...","[{'id': 'session-3788-num-1-para-13', 'type': ...","[{'entity': {'id': 'P0005557', 'name': 'Dorsse..."
4,session-3788-num-1-resolution-13,resolution,"{'session_weekday': 'vrijdag', 'session_month'...","[{'id': 'session-3788-num-1-para-14', 'type': ...","[{'entity': {'id': 'H0005928', 'name': 'straat..."


In [5]:
resolutie_df['paragraphs'][0]

[{'id': 'session-3788-num-1-para-2',
  'type': 'republic_paragraph',
  'metadata': {'page_ids': ['NL-HaNA_1.01.02_3788_0046-page-90'],
   'text_page_num': [2],
   'page_num': [90]},
  'text': 'ONtfangen een Missive van den Resident Spina, geschreven te Franckfort den aght en twintighsten der voorlede maandt, houdende advertentie. WAAR op geen resolutie is gevallen.'}]

In [12]:
resolutie_df['metadata'][0]

{'session_weekday': 'vrijdag',
 'session_month': 1,
 'type': 'resolution',
 'resolution_type': 'ordinaris',
 'id': 'session-3788-num-1-resolution-1',
 'session_year': 1733,
 'session_day': 2,
 'session_date': '1733-01-02',
 'session_id': 'session-3788-num-1',
 'page_num': [90],
 'session_num': 1,
 'page_ids': ['NL-HaNA_1.01.02_3788_0046-page-90'],
 'inventory_num': 3788,
 'source_id': 'session-3788-num-1',
 'text_page_num': [2],
 'proposition_type': ['missive'],
 'resolution_num': 1}

## Proof of Concept: Convert a sample to Republic data model

This PoC converts a small sample of `resolutie_df` using `republic_document_model` from `republic_latest`.
If external dependencies are missing, it falls back to a lightweight normalized record shape so we can still inspect structure and errors.

In [9]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import pandas as pd

# Ensure data is available if earlier cells were not run in this kernel.
if "resolutie_df" not in globals():
    resolutie_file = Path("/Users/rikhoekstra/Downloads/resolutions-version_2025-02-20.jsonl.gz")
    resolutie_df = pd.read_json(resolutie_file, lines=True, compression="gzip")

@dataclass
class SimpleParagraph:
    id: str | None
    type: str | list[str] | None
    metadata: dict[str, Any]
    text: str
    text_region_ids: list[str]
    line_ranges: list[dict[str, Any]]
    scan_versions: list[dict[str, Any]]

@dataclass
class SimpleResolution:
    id: str | None
    type: str | list[str] | None
    metadata: dict[str, Any]
    paragraphs: list[SimpleParagraph]
    paragraph_texts: list[str]
    resolutions_text: str
    labels: list[Any]
    linked_text_regions: list[dict[str, Any]]
    evidence: list[Any]

def _ensure_list(v: Any) -> list[Any]:
    if v is None:
        return []
    return v if isinstance(v, list) else [v]

# Flatten converted objects for search/export, including date
def _extract_date(obj) -> Any:
    meta = getattr(obj, "metadata", {}) or {}
    if not isinstance(meta, dict):
        return None
    if "session_date" in meta and meta["session_date"]:
        return meta["session_date"]
    date_block = meta.get("date")
    if isinstance(date_block, dict):
        if date_block.get("session_date"):
            return date_block.get("session_date")
        if date_block.get("date_string"):
            return date_block.get("date_string")
    for k in ("resolution_date", "date", "year"):
        if meta.get(k):
            return meta.get(k)
    return None

def to_simple_paragraph(paragraph_json: dict[str, Any]) -> SimpleParagraph:
    return SimpleParagraph(
        id=paragraph_json.get("id"),
        type=paragraph_json.get("type"),
        metadata=paragraph_json.get("metadata") or {},
        text=paragraph_json.get("text") or "",
        text_region_ids=_ensure_list(paragraph_json.get("text_region_ids")),
        line_ranges=_ensure_list(paragraph_json.get("line_ranges")),
        scan_versions=_ensure_list(paragraph_json.get("scan_versions")),
    )

def to_simple_resolution(rec: dict[str, Any]) -> SimpleResolution:
    paragraphs = [to_simple_paragraph(p) for p in _ensure_list(rec.get("paragraphs")) if isinstance(p, dict)]
    paragraph_texts = [p.text for p in paragraphs if p.text]
    resolutions_text = "\n\n".join(paragraph_texts)
    return SimpleResolution(
        id=rec.get("id"),
        type=rec.get("type"),
        metadata=rec.get("metadata") or {},
        paragraphs=paragraphs,
        paragraph_texts=paragraph_texts,
        resolutions_text=resolutions_text,
        labels=_ensure_list(rec.get("labels")),
        linked_text_regions=_ensure_list(rec.get("linked_text_regions")),
        evidence=_ensure_list(rec.get("evidence")),
    )

sample_n = min(20, len(resolutie_df))
sample_df = resolutie_df.sample(n=sample_n, random_state=42).copy()

converted_objects: list[SimpleResolution] = []
errors: list[dict[str, Any]] = []

for rec in sample_df.to_dict("records"):
    try:
        converted_objects.append(to_simple_resolution(rec))
    except Exception as e:
        errors.append({
            "id": rec.get("id"),
            "type": rec.get("type"),
            "error": repr(e),
        })

type_counts = pd.Series([type(obj).__name__ for obj in converted_objects]).value_counts().rename_axis("object_type").reset_index(name="count")

summary = {
    "sample_n": sample_n,
    "converted_ok": len(converted_objects),
    "failed": len(errors),
    "using_republic_model": False,
    "import_error": None,
    "model_variant": "SimpleResolution (pagexml-free)",
}

summary_df = pd.DataFrame([summary])
error_df = pd.DataFrame(errors)

display(summary_df)
display(type_counts)
display(error_df.head(10))

,sample_n,converted_ok,failed,using_republic_model,import_error,model_variant
0,20,20,0,False,None,SimpleResolution (pagexml-free)


,object_type,count
0,SimpleResolution,20


""


In [10]:
# Quick inspection of converted sample objects with paragraph texts
if converted_objects:
    object_preview = []
    for obj in converted_objects[:10]:
        object_preview.append({
            "object_type": type(obj).__name__,
            "id": getattr(obj, "id", None),
            "type": getattr(obj, "type", None),
            "date": _extract_date(obj),
            "paragraph_texts_preview": (getattr(obj, "paragraph_texts", [])[:2]),
            "resolutions_text_preview": (getattr(obj, "resolutions_text", "")[:300]),
            "resolutions_text_len": len(getattr(obj, "resolutions_text", "")),
        })
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(object_preview))
else:
    print("No converted objects to preview.")

object_type                                 id        type  \
0  SimpleResolution  session-3116-num-110-resolution-7  resolution   
1  SimpleResolution   session-3296-num-65-resolution-4  resolution   
2  SimpleResolution  session-3762-num-40-resolution-26  resolution   
3  SimpleResolution   session-3184-num-27-resolution-2  resolution   
4  SimpleResolution   session-3291-num-39-resolution-4  resolution   
5  SimpleResolution  session-3766-num-222-resolution-5  resolution   
6  SimpleResolution  session-3255-num-275-resolution-2  resolution   
7  SimpleResolution  session-4614-num-133-resolution-3  resolution   
8  SimpleResolution  session-3120-num-226-resolution-8  resolution   
9  SimpleResolution   session-3798-num-33-resolution-7  resolution   

         date  \
0  1586-06-05   
1  1677-09-08   
2  1707-02-11   
3  1625-01-30   
4  1675-02-09   
5  1711-08-13   
6  1649-11-13   
7  1703-06-09   
8  1588-09-01   
9  1743-02-04   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [17]:

flat_rows = []
for obj in converted_objects:
    flat_rows.append({
        "id": getattr(obj, "id", None),
        "type": getattr(obj, "type", None),
        "proposition_type": getattr(obj, "metadata", {}).get("proposition_type"),
        "date": _extract_date(obj),
        "paragraph_texts": getattr(obj, "paragraph_texts", []),
        "resolutions_text": getattr(obj, "resolutions_text", ""),
    })

resolutions_flat_df = pd.DataFrame(flat_rows)
with pd.option_context("display.max_colwidth", 240):
    display(resolutions_flat_df.head(10))
resolutions_flat_df.shape

,id,type,proposition_type,date,paragraph_texts,resolutions_text
0,session-3116-num-110-resolution-7,resolution,[resolutie],1586-06-05,"[Is geresolueert, dat eenige gedeputeerde vuyt deze vergaderinge sullen worden gevueght byde voers verthoonders, omme gesamenderhant heur doleantien aen te geuen den Raedt van State, ende vande zelue te verstaen hoe deze zaecke by zyne ...","Is geresolueert, dat eenige gedeputeerde vuyt deze vergaderinge sullen worden gevueght byde voers verthoonders, omme gesamenderhant heur doleantien aen te geuen den Raedt van State, ende vande zelue te verstaen hoe deze zaecke by zyne E..."
1,session-3296-num-65-resolution-4,resolution,[rekest],1677-09-08,"[Is ter Vergaderinge gelesen de requeste van Nicolaes Libert, Commissaris ende burger van Luijck, versoeckende dat haer ho:Mo: den Suppliant gelieven te verleenen paspoort voor den tijt van twee maenden, om wegens sijn siecte met een kn...","Is ter Vergaderinge gelesen de requeste van Nicolaes Libert, Commissaris ende burger van Luijck, versoeckende dat haer ho:Mo: den Suppliant gelieven te verleenen paspoort voor den tijt van twee maenden, om wegens sijn siecte met een kne..."
2,session-3762-num-40-resolution-26,resolution,[rapport],1707-02-11,"[Is gehoort het rapport van de Heeren van Welderen, ende andere haer Hoogh Mogende Gedeputeerden tot de saecken van de Nalatenschap van sijne Majesteyt den Koningh van Groot Brittannien, glorieuser gedachtenisse, hebbende, in gevolge en...","Is gehoort het rapport van de Heeren van Welderen, ende andere haer Hoogh Mogende Gedeputeerden tot de saecken van de Nalatenschap van sijne Majesteyt den Koningh van Groot Brittannien, glorieuser gedachtenisse, hebbende, in gevolge end..."
3,session-3184-num-27-resolution-2,resolution,[missive],1625-01-30,"[Ontfangen een missive vant Collegie ter Admiraliteyt tot Rotterdam aldaer geschreven den xxixen. deses, daerby deselve versoucken te verstaen haer ho:Mo: resolutie, hoe sich de Licentmeesters sullen hebben te gedraegen, aengaende den u...","Ontfangen een missive vant Collegie ter Admiraliteyt tot Rotterdam aldaer geschreven den xxixen. deses, daerby deselve versoucken te verstaen haer ho:Mo: resolutie, hoe sich de Licentmeesters sullen hebben te gedraegen, aengaende den uy..."
4,session-3291-num-39-resolution-4,resolution,[onbekend],1675-02-09,"[De resolutien gisteren genomen sijn gelesen ende geresumeert, gelijck oock gersumeert ende gearresteert sijnde de depesches daer uijt resulterende.]","De resolutien gisteren genomen sijn gelesen ende geresumeert, gelijck oock gersumeert ende gearresteert sijnde de depesches daer uijt resulterende."
5,session-3766-num-222-resolution-5,resolution,[missive],1711-08-13,"[ONtfangen een Missive van den Secretaris Runckel, geschreven tot Schafhuysen den sesden deser loopende maendt, houdende advertentie. WAAR op geen resolutie is gevallen.]","ONtfangen een Missive van den Secretaris Runckel, geschreven tot Schafhuysen den sesden deser loopende maendt, houdende advertentie. WAAR op geen resolutie is gevallen."
6,session-3255-num-275-resolution-2,resolution,[onbekend],1649-11-13,"[Item t' concept vanden brieff aenden heer Hertoch van Nieuburch in saecken vanden rector Curtenius, spruijtende uijt haer Hooch Mo: resolutien van gisteren.]","Item t' concept vanden brieff aenden heer Hertoch van Nieuburch in saecken vanden rector Curtenius, spruijtende uijt haer Hooch Mo: resolutien van gisteren."
7,session-4614-num-133-resolution-3,resolution,[missive],1703-06-09,"[Ontfangen een missive vanden Generael Major Goor, Missive voor. bedencking geschreven in het Campement bij Buhl den vierden voor een belegeringe van deser, houdende advertentie, ende onder anderen van Landauw, Bisschop van het passeren...","Ontfangen een missive vanden Generael Major Goor, Missive voor. bedencking geschreven in het Campement bij Buhl den vierden voor een belegeringe van deser, houdende advertentie, ende onder anderen van Landauw, Bisschop van het passeren ..."
8,sess

(20, 6)

In [30]:
import pandas as pd

work = resolutie_df.copy()

# keep only fields you need
flat = work[["id", "type", "metadata", "paragraphs"]].copy()

# paragraph_texts: list[str]
flat["paragraph_texts"] = flat["paragraphs"].apply(
    lambda ps: [
        p.get("text", "")
        for p in (ps if isinstance(ps, list) else [])
        if isinstance(p, dict) and p.get("text")
    ]
)

# concatenated text
flat["resolutions_text"] = flat["paragraph_texts"].str.join("\n\n")

# date extraction from metadata
def extract_date(meta):
    if not isinstance(meta, dict):
        return None
    if meta.get("session_date"):
        return meta["session_date"]
    d = meta.get("date")
    if isinstance(d, dict):
        return d.get("session_date") or d.get("date_string")
    return meta.get("resolution_date") or meta.get("date") or meta.get("year")

flat["date"] = flat["metadata"].apply(extract_date)
flat["weekday"] = flat["metadata"].apply(lambda meta: meta.get('session_weekday') if isinstance(meta, dict) else None)
flat['year'] = flat["metadata"].apply(lambda meta: meta.get("session_year") if isinstance(meta, dict) else '') # int later on
# final table
resolutions_flat_df = flat[["id", "type", "date", "year", "weekday", "paragraph_texts", "resolutions_text"]]

In [29]:
flat["metadata"].apply(lambda meta: meta.get('session_weekday') if isinstance(meta, dict) else None)

0         vrijdag
1         vrijdag
2         vrijdag
3         vrijdag
4         vrijdag
           ...   
692151    dinsdag
692152    dinsdag
692153    dinsdag
692154    dinsdag
692155    dinsdag
Name: metadata, Length: 692156, dtype: object

In [31]:
print(resolutions_flat_df.shape)
resolutions_flat_df.sample(10)

(692156, 7)


,id,type,date,year,weekday,paragraph_texts,resolutions_text
166746,session-4596-num-191-resolution-1,resolution,1690-10-03,1690,dinsdag,[Ontfangen een missive van heere van Amerongen...,"Ontfangen een missive van heere van Amerongen,..."
495859,session-3296-num-113-resolution-6,resolution,1677-10-27,1677,woensdag,[Is ter Vergaderinge gelesen de requeste van J...,Is ter Vergaderinge gelesen de requeste van Je...
293132,session-3806-num-167-resolution-8,resolution,1751-06-21,1751,maandag,[ONtfangen een Missive van den Amanuensis van ...,ONtfangen een Missive van den Amanuensis van w...
220543,session-3260-num-100-resolution-7,resolution,1654-04-16,1654,donderdag,[Sijnde ter Vergaderinge gelesen de Requeste T...,Sijnde ter Vergaderinge gelesen de Requeste Th...
208706,session-456-num-209-resolution-5,resolution,1704-09-09,1704,dinsdag,[Is gehoort het rapport van de Heeren van Esse...,Is gehoort het rapport van de Heeren van Essen...
514975,session-3802-num-209-resolution-5,resolution,1747-08-01,1747,dinsdag,[IS ter Vergaderinge geleesen een Memorie van ...,IS ter Vergaderinge geleesen een Memorie van d...
49576,session-3305-num-83-resolution-10,resolution,1682-03-31,1682,dinsdag,[Ontfangen een missive vande heeren van Citter...,Ontfangen een missive vande heeren van Citters...
554954,session-3183-num-49-resolution-4,resolution,1624-02-26,1624,maandag,[Is gelesen het concept vande missive die aen ...,Is gelesen het concept vande missive die aen d...
135934,session-3266-num-206-resolution-6,resolution,1660-08-05,1660,donderdag,[Sijnde ter Vergaderinge gelesen de Requeste v...,Sijnde ter Vergaderinge gelesen de Requeste va...
628933,session-3850-num-58-resolution-4,resolution,1788-02-28,1788,donderdag,[ONtfangen een Missive van het Collegie ter Ad...,ONtfangen een Missive van het Collegie ter Adm...


In [ ]:
# only execute this if i really need to, otherwise load it from this oarquet file later on

import json

# Build a Parquet-safe export table from the already flattened DataFrame.
resolutions_parquet_df = resolutions_flat_df.copy()

resolutions_parquet_df["type"] = resolutions_parquet_df["type"].apply(
    lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v
)
resolutions_parquet_df["paragraph_texts"] = resolutions_parquet_df["paragraph_texts"].apply(
    lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v
)

resolutions_parquet_df.to_parquet("resolutions_flat.parquet", index=False)
resolutions_flat_df.to_csv("resolutions_flat.csv", sep="\t", index=False)

print("Wrote resolutions_flat.parquet and resolutions_flat.csv")
resolutions_parquet_df.dtypes

Wrote resolutions_flat.parquet and resolutions_flat.csv


id                  object
type                object
date                object
year                 int64
weekday             object
paragraph_texts     object
resolutions_text    object
dtype: object